# Ch 17　SQL agent：讓 agent 安全查資料庫

> **本 notebook 用內建 `sqlite3` 建真資料庫，SQL「生成」用離線假實作，免 API key。** 重點是流程與**安全防線**。

In [ ]:
!uv pip install -q langgraph

In [ ]:
import sqlite3

# 開發用玩具資料庫（production 請用最小權限、唯讀帳號！）
conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.executescript("""
    CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price INTEGER, stock INTEGER);
    INSERT INTO products (name, price, stock) VALUES
        ('滑鼠', 590, 120), ('鍵盤', 1290, 45), ('螢幕', 6990, 8);
""")
print("資料庫建好了")

## 把讀取包成「唯讀」工具：防線寫進程式，不只靠 prompt

In [ ]:
def list_tables() -> str:
    rows = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    return ", ".join(r[0] for r in rows)

def get_schema(table: str) -> str:
    rows = conn.execute(f"PRAGMA table_info({table})").fetchall()
    return "\n".join(f"{r[1]} {r[2]}" for r in rows)

def run_query(query: str) -> str:
    # 最起碼的防線：只准 SELECT（把「唯讀」寫進程式，而非拜託模型乖乖的）
    if not query.strip().lower().startswith("select"):
        return "❌ 錯誤：只允許 SELECT 查詢。"
    if ";" in query.strip().rstrip(";"):
        return "❌ 錯誤：不允許多語句。"
    return str(conn.execute(query).fetchall())

print("表：", list_tables())
print("schema：", get_schema("products").replace(chr(10), " | "))

## 跑一次流程：list → query；並驗證防線擋得住惡意查詢

In [ ]:
# 離線「SQL 生成」：真實情境由 LLM 依問題 + schema 產生（這裡寫死示範流程）
def generate_sql_offline(question: str) -> str:
    if "庫存" in question and "最少" in question:
        return "SELECT name, stock FROM products ORDER BY stock ASC LIMIT 1"
    return "SELECT name, price FROM products ORDER BY price DESC"

q = "庫存最少的商品是哪個？"
sql = generate_sql_offline(q)
print(f"問題：{q}\n產生 SQL：{sql}\n結果：{run_query(sql)}\n")

# 防線測試：這些都該被擋下
print("DROP 攻擊 →", run_query("DROP TABLE products"))
print("多語句  →", run_query("SELECT 1; DELETE FROM products"))

## 重點回顧

第一原則是**安全**：DB 帳號最小權限、唯讀、把防線寫進程式（工具內擋非 SELECT / 多語句）。把離線 `generate_sql_offline` 換成 `bind_tools` 的模型，並用 system prompt 要求「先 list_tables → get_schema → 檢查 → 執行」，即是線上版。